In [ ]:
%pip install -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

In [ ]:
from src.config import get_config_version, get_data_root, get_env, get_spark

spark = get_spark()

print(f"ambiente .......: {get_env()}")
print(f"raiz de dados ..: {get_data_root()}")
print(f"config version .: {get_config_version()}")
print(f"spark version ..: {spark.version}")
print(f"timezone .......: {spark.conf.get('spark.sql.session.timeZone')}")

In [ ]:
dbutils.fs.mkdirs("dbfs:/tmp/novarota")
display(dbutils.fs.ls("dbfs:/tmp/"))

In [ ]:
from scripts.gerar_massa import gerar_massa

gerar_massa()

In [ ]:
from src.pipeline import executar

resumo = executar()

In [ ]:
display(dbutils.fs.ls(f"{get_data_root().replace('/dbfs', 'dbfs:')}/gold"))

In [ ]:
from src.config import resolve_target

for tabela in (
    "gold_transacoes_cartao",
    "gold_resumo_cliente_mes",
    "gold_ranking_estabelecimento",
    "gold_alertas_qualidade",
):
    linhas = spark.read.format("delta").load(resolve_target("gold", tabela)).count()
    print(f"{tabela:32s} {linhas:7d} linhas")

In [ ]:
display(
    spark.read.format("delta")
    .load(resolve_target("gold", "gold_alertas_qualidade"))
    .orderBy("entidade", "tipo")
)

In [ ]:
from pathlib import Path

from src.transformations.gold import registrar_views, registrar_views_gold

registrar_views()
registrar_views_gold()

for arquivo in sorted(Path("sql/analises").glob("*.sql")):
    print(f"\n{'=' * 72}\n{arquivo.name}\n{'=' * 72}")
    display(spark.sql(arquivo.read_text(encoding="utf-8")))

In [ ]:

%sql

CREATE SCHEMA IF NOT EXISTS novarota_gold;

CREATE TABLE IF NOT EXISTS novarota_gold.gold_transacoes_cartao

USING DELTA LOCATION 'dbfs:/tmp/novarota/gold/gold_transacoes_cartao';

CREATE TABLE IF NOT EXISTS novarota_gold.gold_resumo_cliente_mes

USING DELTA LOCATION 'dbfs:/tmp/novarota/gold/gold_resumo_cliente_mes';

CREATE TABLE IF NOT EXISTS novarota_gold.gold_ranking_estabelecimento

USING DELTA LOCATION 'dbfs:/tmp/novarota/gold/gold_ranking_estabelecimento';

CREATE TABLE IF NOT EXISTS novarota_gold.gold_alertas_qualidade

USING DELTA LOCATION 'dbfs:/tmp/novarota/gold/gold_alertas_qualidade';

In [ ]:
%sql
SELECT nome_estabelecimento, cidade_estabelecimento, valor_liquido, rank_valor
FROM novarota_gold.gold_ranking_estabelecimento
ORDER BY rank_valor
LIMIT 10;